#### Joindre des données provenant de différentes sources (fichiers locaux, S3, bases de données externes comme PostgreSQL via des extensions).

- Joindre un fichier Parquet en local avec un fichier sur un url distante
- Concaténer des extracts mensuels en une seule requête SQL
- Joindre des données en mémoire avec un fichier parquet sur un storage Account Azure


In [1]:
import duckdb as d, pandas as pd, polars as pl, time

- Joindre un fichier Parquet en local avec un fichier sur un url distante


In [8]:
local_parquet_file = '../00_data_sources/yellow_tripdata_2023-01.parquet'
remote_parquet_file = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet'
trips_data = d.sql(
    f"""
    SELECT *
    FROM read_parquet('{local_parquet_file}') AS t1
    UNION ALL
    SELECT *
    FROM read_parquet('{remote_parquet_file}') AS t2
    """
)

In [9]:
nb_trips_per_month = d.sql(
    """
    SELECT
        strftime(tpep_pickup_datetime , '%Y-%m') AS trip_month,
        COUNT(*) AS monthly_trips,
    FROM trips_data
    WHERE trip_month IN ('2023-01', '2023-02')
    GROUP BY trip_month
    ORDER BY trip_month;
    """
)
print(nb_trips_per_month)


┌────────────┬───────────────┐
│ trip_month │ monthly_trips │
│  varchar   │     int64     │
├────────────┼───────────────┤
│ 2023-01    │       3066726 │
│ 2023-02    │       2913910 │
└────────────┴───────────────┘



- Concaténer des données mensuelles en une seule requête SQL

In [11]:
parquet_files_pattern = "../00_data_sources/yellow_tripdata_2025-*.parquet"

start_time = time.time()

full_taxi_data_2025 = d.sql(
    f"""
    SELECT * 
    FROM read_parquet('{parquet_files_pattern}')
    """
)

nb_trips_2025 = d.sql(
    """
    SELECT COUNT(*) AS total_trips_2025
    FROM full_taxi_data_2025 ;
    """
)

execution_time_duckdb = time.time() - start_time
print(nb_trips_2025)
print(f"Exécution DuckDB: {execution_time_duckdb:.4f} secondes")


┌──────────────────┐
│ total_trips_2025 │
│      int64       │
├──────────────────┤
│         35807453 │
└──────────────────┘

Exécution DuckDB: 0.0051 secondes


In [17]:
print(full_taxi_data_2025)

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │       double       │
├──────

- Joindre des données en mémoire avec un fichier parquet sur un storage Account Azure


In [21]:
#Joindre des données en mémoire avec un fichier parquet sur un storage Account Azure
blob_sas_url = 'https://saduckdb.blob.core.windows.net/contduckdb/taxi_zone_lookup.csv?'
blob_sas_token = 'sp=r&st=2025-12-22T09:00:57Z&se=2025-12-22T17:15:57Z&spr=https&sv=2024-11-04&sr=b&sig=1If6C6hVMJc71qwAIUTLjUGsqIzZgq1OaxX1GXo46A0%3D'

# Charger la table des zones
trips_with_zones = d.sql(
    f"""
    SELECT 
        t.*,
        p.Zone AS Pickup_Zone
    FROM full_taxi_data_2025 AS t
    LEFT JOIN (
    SELECT * FROM read_csv_auto('{blob_sas_url}{blob_sas_token}')
    ) AS p
    ON t.PULocationID = p.LocationID
"""
)

print(trips_with_zones)

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┬───────────────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │          Pickup_Zone          │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │  